# 25 - Benchmark Triton Runtime

Benchmark limpo de throughput/latência para o runtime packed ternário com Triton.

Objetivos:

1. Separar warmup/compilação do tempo medido.
2. Medir throughput com repetições por prompt.
3. Registrar p50/p90/p95 de latência por geração.
4. Comparar builds quando houver mais de um diretório disponível.

Este notebook mede operação do runtime. Não substitui benchmark externo de qualidade.


## Reexecução após a auditoria

Os outputs já gravados abaixo são históricos e foram preservados. As células atualizadas ainda precisam de execução no hardware alvo.

O runtime é extraído integralmente de um único commit: use `RPT_RUNTIME_CHECKOUT` com um checkout Git do RPT; `HEAD` é congelado em um SHA completo. Arquivos locais não commitados não são utilizados. Fora de um checkout, configure `RPT_RUNTIME_REV` com os 40 caracteres do commit publicado; não há download de `main` nem preenchimento parcial de arquivos ausentes. A revisão precisa conter `artifacts.py`, `execution.py` e `metrics.py`.

Os hashes e a revisão são gravados em `runtime_provenance.json`. As medições Triton falham se houver fallback. Essa verificação comprova o backend observado e não comprova ganho de velocidade, generalização ou compatibilidade com outro acelerador. A suíte histórica de 100 perguntas é desenvolvimento, pois foi usada na recuperação. Prewarm nesta sessão não garante cache quente em outra instalação. Instalação de dependências e cópias grandes continuam explícitas nas células históricas; a revisão estática não as executa.


In [ ]:
# Dependências principais.
import os, sys, subprocess, importlib.util, importlib.metadata

required = {
    'torch': 'torch',
    'triton': 'triton',
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'peft': 'peft',
    'safetensors': 'safetensors',
    'sentencepiece': 'sentencepiece',
    'psutil': 'psutil',
    'packaging': 'packaging',
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Instalando dependências ausentes:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

from packaging.version import Version
def installed_version(package_name):
    try:
        return Version(importlib.metadata.version(package_name))
    except importlib.metadata.PackageNotFoundError:
        return None

torchao_version = installed_version('torchao')
if torchao_version is not None:
    print(f'Removendo torchao para evitar warnings/side effects no benchmark: {torchao_version}')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

import torch
import psutil

def human_bytes(n):
    n = float(n)
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024.0 or unit == 'TB':
            return f'{n:.2f} {unit}'
        n /= 1024.0

print('Dependências prontas.')


In [ ]:
# Bootstrap Colab/local e paths oficiais.
from pathlib import Path
import json, time, gc, shutil, platform

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'packed_recovery'
BUILD_DIR = PROJECT_ROOT / 'build'
RUNTIME_DIR = PROJECT_ROOT / 'runtime' / 'packed_recovered_l78'
DIST_DIR = BUILD_DIR / 'distributable_packed_recovered_l78'
DATA_PATH = PROJECT_ROOT / 'data' / 'gemma4_100_questions.json'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BUILD_DIR.mkdir(parents=True, exist_ok=True)
DIST_DIR.mkdir(parents=True, exist_ok=True)
if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RUNTIME_DIR =', RUNTIME_DIR)
print('DIST_DIR =', DIST_DIR)

In [ ]:
# Build with one immutable runtime source; never recover a partial/mixed runtime.
STUDENT_DIR = CHECKPOINTS_DIR / 'student_pruned'
ADAPTER_DIR = CHECKPOINTS_DIR / 'student_packed_ternary_recovered_l78_t4_resume'
PACKED_SOURCE_DIR = next((p for p in [EXPORTS_DIR / 'ternary_mlp_l78', DEPLOY_DIR]
                          if (p / 'manifest.json').exists() and (p / 'mlp_ternary_2bit.bin').exists()), None)

# All runtime files come from one immutable Git revision, never mixed Drive/main files.
import hashlib
import re
import subprocess
import tempfile
import urllib.request

RUNTIME_FILES = [
    'README.md', 'requirements.txt', 'loader.py', 'run_inference.py',
    'native_loader.py', 'run_native_inference.py', 'triton_loader.py',
    'run_triton_inference.py', 'package_manifest.template.json',
    'artifacts.py', 'execution.py', 'metrics.py',
]
# The development evaluator is frozen alongside the runtime, from the same commit.
RUNTIME_SUPPORT_FILES = {'evaluation_policy.py': 'validation/evaluation.py'}
RUNTIME_FILES += list(RUNTIME_SUPPORT_FILES)
RUNTIME_MODULES = ['triton_loader', 'native_loader', 'loader', 'artifacts', 'execution', 'metrics', 'evaluation_policy']
RUNTIME_REPO_PREFIX = 'gemma4_theory/runtime/packed_recovered_l78'
RUNTIME_SOURCE_DIR = None
RUNTIME_PROVENANCE = None

def ensure_runtime_templates():
    global RUNTIME_SOURCE_DIR, RUNTIME_PROVENANCE
    if RUNTIME_SOURCE_DIR is not None:
        for filename, expected in RUNTIME_PROVENANCE['file_sha256'].items():
            if hashlib.sha256((RUNTIME_SOURCE_DIR / filename).read_bytes()).hexdigest() != expected:
                raise RuntimeError(f'Pinned runtime snapshot changed: {filename}')
        return RUNTIME_FILES[:] if RUNTIME_PROVENANCE['source_kind'] == 'github_commit' else []

    checkout = Path(os.environ.get('RPT_RUNTIME_CHECKOUT', str(PROJECT_ROOT.parent))).expanduser().resolve()
    revision = os.environ.get('RPT_RUNTIME_REV', '').strip().lower()
    if revision and re.fullmatch(r'[0-9a-f]{40}', revision) is None:
        raise ValueError('RPT_RUNTIME_REV must be a complete 40-character Git commit SHA')
    try:
        top = subprocess.run(['git', '-C', str(checkout), 'rev-parse', '--show-toplevel'],
                             check=True, capture_output=True, text=True).stdout.strip()
        local_checkout = Path(top).resolve() == checkout
    except (OSError, subprocess.CalledProcessError):
        local_checkout = False

    if local_checkout:
        if not revision:
            revision = subprocess.run(['git', '-C', str(checkout), 'rev-parse', 'HEAD'],
                                      check=True, capture_output=True, text=True).stdout.strip().lower()
        resolved = subprocess.run(['git', '-C', str(checkout), 'rev-parse', '--verify', revision + '^{commit}'],
                                  check=True, capture_output=True, text=True).stdout.strip().lower()
        if resolved != revision or re.fullmatch(r'[0-9a-f]{40}', revision) is None:
            raise RuntimeError('Runtime revision is not an immutable commit')
        source_kind = 'local_git_commit'
    else:
        if not revision:
            raise RuntimeError('Use a local RPT Git checkout or set RPT_RUNTIME_REV to a full published commit SHA. '
                               'Automatic download from main and partial runtime refresh are disabled.')
        source_kind = 'github_commit'

    # Read every file first. An incomplete revision fails before any build is replaced.
    contents = {}
    for filename in RUNTIME_FILES:
        repo_path = RUNTIME_SUPPORT_FILES.get(filename, f'{RUNTIME_REPO_PREFIX}/{filename}')
        if local_checkout:
            try:
                contents[filename] = subprocess.run(['git', '-C', str(checkout), 'show', f'{revision}:{repo_path}'],
                                                    check=True, capture_output=True).stdout
            except subprocess.CalledProcessError as exc:
                raise RuntimeError(f'Runtime revision {revision} is incomplete: {repo_path}') from exc
        else:
            url = f'https://raw.githubusercontent.com/cesarfavero/RPT/{revision}/{repo_path}'
            with urllib.request.urlopen(url, timeout=30) as response:
                contents[filename] = response.read()
    snapshot_parent = PROJECT_ROOT / 'build' / 'runtime_snapshots'
    snapshot_parent.mkdir(parents=True, exist_ok=True)
    snapshot = Path(tempfile.mkdtemp(prefix=revision[:12] + '-', dir=snapshot_parent))
    try:
        for filename, content in contents.items():
            (snapshot / filename).write_bytes(content)
        provenance = {'source_kind': source_kind, 'git_revision': revision,
                      'working_tree_used': False,
                      'file_sha256': {name: hashlib.sha256(data).hexdigest() for name, data in contents.items()}}
        (snapshot / 'runtime_provenance.json').write_text(json.dumps(provenance, indent=2) + '\n', encoding='utf-8')
    except BaseException:
        shutil.rmtree(snapshot, ignore_errors=True)
        raise
    RUNTIME_SOURCE_DIR, RUNTIME_PROVENANCE = snapshot, provenance
    return RUNTIME_FILES[:] if source_kind == 'github_commit' else []

def activate_runtime_directory(runtime_dir):
    runtime_dir = Path(runtime_dir).resolve()
    provenance = json.loads((runtime_dir / 'runtime_provenance.json').read_text(encoding='utf-8'))
    if provenance != RUNTIME_PROVENANCE:
        raise RuntimeError('Bundle runtime revision differs from the frozen notebook runtime')
    for filename, expected in provenance['file_sha256'].items():
        if hashlib.sha256((runtime_dir / filename).read_bytes()).hexdigest() != expected:
            raise RuntimeError(f'Bundle runtime file hash mismatch: {filename}')
    runtime_str = str(runtime_dir)
    if runtime_str in sys.path:
        sys.path.remove(runtime_str)
    sys.path.insert(0, runtime_str)
    for module_name in RUNTIME_MODULES:
        sys.modules.pop(module_name, None)
    import importlib
    importlib.invalidate_caches()

def copytree_clean(src, dst):
    src, dst = Path(src), Path(dst)
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))

def copy_file_clean(src, dst):
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    shutil.copy2(src, dst)

def ensure_distributable():
    if PACKED_SOURCE_DIR is None:
        raise FileNotFoundError('Packed source not found in exports/ternary_mlp_l78 or deploy')
    downloaded = ensure_runtime_templates()
    required_sources = [STUDENT_DIR, ADAPTER_DIR, RUNTIME_SOURCE_DIR]
    missing = [str(path) for path in required_sources if not path.exists()]
    if missing:
        raise FileNotFoundError('Missing sources: ' + ', '.join(missing))
    for src, dst in [(STUDENT_DIR, DIST_DIR / 'student_pruned'),
                     (ADAPTER_DIR, DIST_DIR / 'adapter_recovered_l78'),
                     (RUNTIME_SOURCE_DIR, DIST_DIR / 'runtime')]:
        copytree_clean(src, dst)
    (DIST_DIR / 'packed').mkdir(parents=True, exist_ok=True)
    copy_file_clean(PACKED_SOURCE_DIR / 'manifest.json', DIST_DIR / 'packed' / 'manifest.json')
    copy_file_clean(PACKED_SOURCE_DIR / 'mlp_ternary_2bit.bin', DIST_DIR / 'packed' / 'mlp_ternary_2bit.bin')
    # Metadata from a previous build does not certify the rebuilt model.
    for stale in ('artifact_checksums.json', 'production_manifest.json', 'prewarm_profile.json', 'package_manifest.json'):
        (DIST_DIR / stale).unlink(missing_ok=True)
    (DIST_DIR / 'README.md').write_text('RPT packed runtime build\nRuntime revision: ' + RUNTIME_PROVENANCE['git_revision'] + '\nQuality has not been revalidated by packaging. The historical 100-question suite is development data.\n', encoding='utf-8')
    return downloaded

runtime_downloaded = ensure_distributable()
runtime_dist = DIST_DIR / 'runtime'
activate_runtime_directory(runtime_dist)
print('distributable ready:', DIST_DIR)
print('runtime revision:', RUNTIME_PROVENANCE['git_revision'])
print('runtime files:', RUNTIME_FILES)

def synchronize_model_devices(model):
    # PyTorch may expose non-NVIDIA devices through the CUDA interface.
    # Synchronize the actual tensor devices, not only the current device.
    from itertools import chain
    devices = {tensor.device for tensor in chain(model.parameters(), model.buffers())
               if tensor.device.type == 'cuda'}
    for device in sorted(devices, key=str):
        torch.cuda.synchronize(device)


In [ ]:
# Configuração do benchmark.
BENCH_OUT_DIR = RESULTS_DIR / 'runtime_benchmarks'
BENCH_OUT_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_QUESTIONS = [
    'Qual a capital da França?',
    'Quanto é 12 x 8?',
    'Quem desenvolveu a teoria da relatividade?',
    'Em Python, qual palavra-chave define uma função?',
    'Se hoje é segunda-feira, que dia será depois de 3 dias?',
    'Qual esporte usa raquete, rede e peteca?',
    'Qual gás as plantas absorvem principalmente para fazer fotossíntese?',
    'Qual comando Linux lista arquivos de um diretório?',
]

WARMUP_ROUNDS = 1
MEASURE_ROUNDS = 3
MAX_NEW_TOKENS = 12
PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'

# DIST_DIR é reconstruído automaticamente na célula anterior.
BUILD_CANDIDATES = [
    DIST_DIR,
    BUILD_DIR / 'packed_recovered_l78',
]
MODEL_DIRS = []
for p in BUILD_CANDIDATES:
    if p.exists() and (p / 'runtime').exists() and (p / 'packed').exists():
        MODEL_DIRS.append(p)
if not MODEL_DIRS:
    raise FileNotFoundError('Nenhum build pronto encontrado após auto-rebuild. Esperado: runtime/ e packed/ em build/distributable_packed_recovered_l78 ou build/packed_recovered_l78')
print('MODEL_DIRS =', [str(p) for p in MODEL_DIRS])
print('runtime_downloaded =', runtime_downloaded)


In [ ]:
import statistics


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def snapshot_memory(label):
    rss = psutil.Process().memory_info().rss
    item = {
        'label': label,
        'rss_bytes': rss,
        'rss_human': human_bytes(rss),
    }
    if torch.cuda.is_available():
        item.update({
            'cuda_device': torch.cuda.get_device_name(0),
            'cuda_allocated_bytes': torch.cuda.memory_allocated(),
            'cuda_reserved_bytes': torch.cuda.memory_reserved(),
            'cuda_max_allocated_bytes': torch.cuda.max_memory_allocated(),
            'cuda_allocated_human': human_bytes(torch.cuda.memory_allocated()),
            'cuda_reserved_human': human_bytes(torch.cuda.memory_reserved()),
            'cuda_max_allocated_human': human_bytes(torch.cuda.max_memory_allocated()),
        })
    return item


def percentile(values, q):
    if not values:
        return None
    if len(values) == 1:
        return values[0]
    vals = sorted(values)
    idx = (len(vals) - 1) * q
    lo = int(idx)
    hi = min(lo + 1, len(vals) - 1)
    frac = idx - lo
    return vals[lo] * (1 - frac) + vals[hi] * frac


def extract_answer(prompt, generated):
    text = generated
    if prompt in text:
        text = text.split(prompt, 1)[-1]
    return text.strip().split('\n')[0].strip()


In [ ]:
# A Triton benchmark must fail if its requested backend cannot execute.
def load_runtime_model(model_dir):
    activate_runtime_directory(Path(model_dir) / 'runtime')
    import importlib
    runtime = importlib.import_module('triton_loader')
    started = time.perf_counter()
    model, tokenizer, report = runtime.load_triton_packed_recovered_l78(
        str(model_dir), token=os.environ.get('HF_TOKEN'),
        verify_sha=True, require_triton=True, fallback_to_pytorch=False,
    )
    if torch.cuda.is_available():
        synchronize_model_devices(model)
    load_elapsed = time.perf_counter() - started
    model.eval()
    runtime.reset_runtime_execution(model)
    # Keep the reporter associated with this loaded class/module identity.
    model._rpt_observed_execution = runtime.runtime_execution_report
    report = dict(report)
    report['selected_loader'] = 'triton_loader.load_triton_packed_recovered_l78'
    report['runtime_source'] = RUNTIME_PROVENANCE
    return model, tokenizer, report, load_elapsed

def observe_runtime_execution(model):
    if torch.cuda.is_available():
        synchronize_model_devices(model)
    return model._rpt_observed_execution(model, require_triton=True)


In [ ]:
def benchmark_model_dir(model_dir):
    cleanup_memory()
    before = snapshot_memory('before_load')
    model, tokenizer, runtime_report, load_elapsed = load_runtime_model(model_dir)
    after = snapshot_memory('after_load')

    warmup_rows = []
    for question in BENCHMARK_QUESTIONS[:WARMUP_ROUNDS]:
        prompt = PROMPT_TEMPLATE.format(question=question)
        inputs = tokenizer(prompt, return_tensors='pt').to(next(model.parameters()).device)
        with torch.no_grad():
            synchronize_model_devices(model)
            t0 = time.perf_counter()
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
            synchronize_model_devices(model)
            elapsed = time.perf_counter() - t0
        generated = tokenizer.decode(out[0], skip_special_tokens=True)
        answer = extract_answer(prompt, generated)
        generated_tokens = max(0, out.shape[-1] - inputs['input_ids'].shape[-1])
        warmup_rows.append({'question': question, 'answer': answer, 'elapsed_sec': elapsed, 'generated_tokens': generated_tokens})

    measure_rows = []
    for question in BENCHMARK_QUESTIONS:
        prompt = PROMPT_TEMPLATE.format(question=question)
        for rep in range(MEASURE_ROUNDS):
            inputs = tokenizer(prompt, return_tensors='pt').to(next(model.parameters()).device)
            with torch.no_grad():
                synchronize_model_devices(model)
                t0 = time.perf_counter()
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
                synchronize_model_devices(model)
                elapsed = time.perf_counter() - t0
            generated = tokenizer.decode(out[0], skip_special_tokens=True)
            answer = extract_answer(prompt, generated)
            generated_tokens = max(0, out.shape[-1] - inputs['input_ids'].shape[-1])
            measure_rows.append({
                'question': question,
                'rep': rep + 1,
                'answer': answer,
                'elapsed_sec': elapsed,
                'generated_tokens': generated_tokens,
                'tokens_per_sec': (generated_tokens / elapsed) if elapsed > 0 and generated_tokens > 0 else None,
            })

    latencies = [r['elapsed_sec'] for r in measure_rows]
    tps = [r['tokens_per_sec'] for r in measure_rows if r['tokens_per_sec'] is not None]
    total_tokens = sum(r['generated_tokens'] for r in measure_rows)
    total_elapsed = sum(r['elapsed_sec'] for r in measure_rows)

    summary = {
        'model_dir': str(model_dir),
        'load_elapsed_sec': load_elapsed,
        'warmup_rounds': WARMUP_ROUNDS,
        'measure_rounds': MEASURE_ROUNDS,
        'num_questions': len(BENCHMARK_QUESTIONS),
        'num_measured_generations': len(measure_rows),
        'total_generated_tokens': total_tokens,
        'total_elapsed_sec': total_elapsed,
        'aggregate_tokens_per_sec': (total_tokens / total_elapsed) if total_elapsed > 0 else None,
        'avg_tokens_per_sec_weighted': (total_tokens / total_elapsed) if total_elapsed > 0 else None,
        'cold_cache_guaranteed': False,
        'timing_scope': 'complete generation excluding load and tokenization',
        'avg_tokens_per_sec_unweighted': statistics.mean(tps) if tps else None,
        'latency_p50_sec': percentile(latencies, 0.50),
        'latency_p90_sec': percentile(latencies, 0.90),
        'latency_p95_sec': percentile(latencies, 0.95),
        'latency_min_sec': min(latencies) if latencies else None,
        'latency_max_sec': max(latencies) if latencies else None,
    }
    runtime_report.update(observe_runtime_execution(model))
    result = {
        'model_dir': str(model_dir),
        'runtime_report': runtime_report,
        'memory_before_load': before,
        'memory_after_load': after,
        'warmup_rows': warmup_rows,
        'measure_rows': measure_rows,
        'summary': summary,
    }
    del model, tokenizer
    cleanup_memory()
    return result


In [ ]:
all_results = []
for model_dir in MODEL_DIRS:
    print('\n' + '=' * 100)
    print('Benchmarking:', model_dir)
    result = benchmark_model_dir(model_dir)
    all_results.append(result)
    print(json.dumps(result['summary'], ensure_ascii=False, indent=2))

comparison = None
if len(all_results) >= 2:
    base = all_results[0]['summary']
    alt = all_results[1]['summary']
    comparison = {
        'base_model_dir': all_results[0]['model_dir'],
        'alt_model_dir': all_results[1]['model_dir'],
        'weighted_tps_speedup_alt_vs_base': (alt['avg_tokens_per_sec_weighted'] / base['avg_tokens_per_sec_weighted']) if base['avg_tokens_per_sec_weighted'] else None,
        'latency_p50_ratio_alt_vs_base': (alt['latency_p50_sec'] / base['latency_p50_sec']) if base['latency_p50_sec'] else None,
        'latency_p95_ratio_alt_vs_base': (alt['latency_p95_sec'] / base['latency_p95_sec']) if base['latency_p95_sec'] else None,
    }

payload = {
    'notebook': '25_benchmark_triton_runtime',
    'purpose': 'Clean throughput/latency benchmark for Triton packed ternary runtime with warmup separated from measurement.',
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'results': all_results,
    'comparison': comparison,
}
out_path = BENCH_OUT_DIR / f"25_benchmark_triton_runtime_{time.strftime('%Y%m%d_%H%M%S')}.json"
out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved', out_path)
print(json.dumps({'comparison': comparison, 'summaries': [r['summary'] for r in all_results]}, ensure_ascii=False, indent=2)[:12000])
